FaceMap Model Implementation

In [54]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# from neural_model import KeypointsNetwork # 'pip install facemap'
from facemap import utils
# from facemap.utils import LoadFacemapData
from utils import compute_varexp, gabor_wavelet, split_data

import matplotlib.pyplot as plt
import numpy as np

from scipy.stats import zscore

import os
import numpy as np

from scipy.io import savemat

In [55]:
## --------------------------------------------------------------------------- CLASS CORE -----------------------------------------------------------
# --------- Linear --> 1D CONV --> ReLU --> Linear --ReLU ===== Latents

class Core(nn.Module):
    def __init__(self, n_in=55, n_kp=None, n_filt=10, kernel_size=201, n_layers=1, n_med=50, n_latents=256, identity=False, relu_wavelets=True, relu_latents=True):
        super().__init__()
        self.n_in = n_in
        self.n_kp = n_in if n_kp is None or identity else n_kp
        self.n_filt = (n_filt // 2) * 2  # must be even for initialization
        self.relu_latents = relu_latents
        self.relu_wavelets = relu_wavelets
        self.kernel_size = kernel_size
        self.n_layers = n_layers
        self.n_latents = n_latents
        self.features = nn.Sequential()

        # Reduces Keypoints Dim (n_in) to Hidden Dim (n_kp) in first layer ("linear0")
        if identity:
            self.features.add_module("linear0", nn.Identity(self.n_in))
        else:
            self.features.add_module(
                "linear0",
                nn.Sequential(
                    nn.Linear(self.n_in, self.n_kp),
                ),
            )


        # Initalize Gabor Wavelets - 1D CONV
        f = np.geomspace(1, 10, self.n_filt // 2).astype("float32")
        gw0 = gabor_wavelet(1, f[:, np.newaxis], 0, n_pts=kernel_size) # First Gabor Wavelet filter into 0 degrees orientation
        gw1 = gabor_wavelet(1, f[:, np.newaxis], np.pi / 2, n_pts=kernel_size) # Second Gabor Wavelet filter into 90 degrees orientation 
        wav_init = np.vstack((gw0, gw1))

        # compute n_filt wavelet features of each one => n_filt * n_kp features
        self.features.add_module(
            "wavelet0",
            nn.Conv1d(
                1,
                self.n_filt,
                kernel_size=kernel_size,
                padding=kernel_size // 2,
                bias=False,
            ),
        )

        self.features[-1].weight.data = torch.from_numpy(wav_init).unsqueeze(1) # Sets Initial Weights of Wavelet Filters

        # Adds Linear layers (n_layers) to features
        for n in range(1, n_layers):
            n_in = self.n_kp * self.n_filt if n == 1 else n_med
            self.features.add_module(f"linear{n}", nn.Sequential(nn.Linear(n_in, n_med)))

        # Adds the final layer (n_med to n_latents)
        n_med = n_med if n_layers > 1 else self.n_filt * self.n_kp
        self.features.add_module("latent", nn.Sequential(nn.Linear(n_med, n_latents)))



    def wavelets(self, x):
        """compute wavelets of keypoints through linear + conv1d + relu layer"""
        # x is (n_batches, time, features)
        out = self.features[0](x.reshape(-1, x.shape[-1]))
        out = out.reshape(x.shape[0], x.shape[1], -1).transpose(2, 1)
        # out is now (n_batches, n_kp, time)
        out = out.reshape(-1, out.shape[-1]).unsqueeze(1)
        # out is now (n_batches * n_kp, 1, time)
        out = self.features[1](out)
        # out is now (n_batches * n_kp, n_filt, time)
        out = out.reshape(-1, self.n_kp * self.n_filt, out.shape[-1]).transpose(2, 1)
        out = out.reshape(-1, self.n_kp * self.n_filt)

        if self.relu_wavelets:
            out = F.relu(out)
        
        # if n_layers > 1, go through more linear layers
        for n in range(1, self.n_layers):
            out = self.features[n + 1](out)
            out = F.relu(out)
        return out

    def forward(self, x=None, wavelets=None):
        """x is (n_batches, time, features)
        sample_inds is (sub_time) over batches
        """
        if wavelets is None:
            wavelets = self.wavelets(x)
        wavelets = wavelets.reshape(-1, wavelets.shape[-1])

        # latent layer
        latents = self.features[-1](wavelets)
        latents = latents.reshape(x.shape[0], -1, latents.shape[-1])
        if self.relu_latents:
            latents = F.relu(latents)
        latents = latents.reshape(-1, latents.shape[-1])
        return latents


In [56]:
## --------------------------------------------------------------------------- CLASS READOUT -----------------------------------------------------------
# --------- Latents --> Session-specific Linear Layer === Neural PCs

class Readout(nn.Module):
    """Linear Layers from Latents (Core Class) ---> Neural PCs with session-specific output dimensions."""
    def __init__(self, out_list, n_latents=256, n_layers=1, n_med=128):
        super().__init__()
        self.n_sessions = len(out_list)
        self.n_latents = n_latents
        self.n_layers = n_layers
        self.n_med = n_med
        self.features = nn.ModuleDict()
        
        # Define separate layers for each session, with dynamic n_out from out_list
        for session_id in range(self.n_sessions):
            session_layers = nn.Sequential()
            for j in range(n_layers):
                n_in = self.n_latents if j == 0 else self.n_med
                n_outc = out_list[session_id] if j == n_layers - 1 else self.n_med
                session_layers.add_module(f"linear{j}", nn.Linear(n_in, n_outc))
                if self.n_layers > 1 and j < self.n_layers - 1:
                    session_layers.add_module(f"relu{j}", nn.ReLU())
            self.features[f"session_{session_id+1}"] = session_layers

    def forward(self, latents, session_id):
        # Forward pass through the specific session's layers
        session_key = f"session_{session_id}"  
        if session_key in self.features:
            return self.features[session_key](latents)
        else:
            print(f"Unexpected Session ID: {session_id}. Returning latents unchanged.")
            return latents  

In [57]:
class KeypointsNetwork(nn.Module):
    """Keypoints to neural PCs / neural activity model"""
    def __init__(self, n_in=55, n_kp=None, n_filt=10, kernel_size=201, n_core_layers=2, n_latents=256, out_list=None, n_out_layers=1, n_med=50, identity=False, relu_wavelets=True, relu_latents=True):
        super().__init__()
        self.core = Core(
            n_in=n_in,
            n_kp=n_kp,
            n_filt=n_filt,
            kernel_size=kernel_size,
            n_layers=n_core_layers,
            n_med=n_med,
            n_latents=n_latents,
            identity=identity,
            relu_wavelets=relu_wavelets,
            relu_latents=relu_latents,
        )
        
        # Ensure out_list is provided and matches the expected number of sessions
        if out_list is None or len(out_list) == 0:
            raise ValueError("out_list must be provided and cannot be empty.")
        
        self.readout = Readout(
            out_list=out_list,  
            n_latents=n_latents,
            n_layers=n_out_layers,
            n_med=n_med
        )

        # Store out_list for session ID validation
        self.valid_session_ids = set(range(len(out_list)))

    def forward(self, x, sample_inds=None):
        
        # Extract session ID (last feature for all samples in the batch)
        session_id = int(x[0, 0, -1].item())  # Access the first time step for the first sample
        x = x[:, :, :-1]
        # print(x.shape)  # To debug
        
        # Run through the Core to get latents
        latents = self.core(x)

        # Sample latents if indices are provided
        if sample_inds is not None:
            latents = latents[sample_inds]
        
        latents = latents.reshape(x.shape[0], -1, latents.shape[-1])  # Reshape latents for output

        # Check if the session_id is valid
        if session_id in self.valid_session_ids:
            # If valid, get prediction from readout
            y_pred = self.readout(latents, session_id=session_id)
            return y_pred, latents
        else:
            # If not valid, return only latents
            print(f"Unexpected Session ID: {session_id}. Returning latents only.")
            return latents
    




EXAMPLE

In [6]:
import pandas as pd

X1 = pd.read_csv('/Users/omarelsayed/Desktop/PhD Research/New Data with All Kinematics/Kinematics Data/YH16_240306_Kinematics_All.csv')
Y1 = pd.read_csv('/Users/omarelsayed/Desktop/PhD Research/New Data with All Kinematics/Neural Data/YH16_240306_R_IRN_Single_Trial.csv')

X2 = pd.read_csv('/Users/omarelsayed/Desktop/PhD Research/New Data with All Kinematics/Kinematics Data/YH16_240307_Kinematics_All.csv')
Y2 = pd.read_csv('/Users/omarelsayed/Desktop/PhD Research/New Data with All Kinematics/Neural Data/YH16_240307_R_IRN_Single_Trial.csv')

In [7]:
# Preprocessing: 
def preprocess_data(X,Y, trial_len, session_id):

    # Drop Trial Type Column 
    if 'Trial Type' in X.columns: 
        X = X.drop(columns = ['Trial Type'])
    if 'Trial Type' in Y.columns: 
        Y = Y.drop(columns = ['Trial Type'])
    
    X = X.to_numpy()
    Y = Y.to_numpy()
    
    # Z - scoring Kinematic Features (X)
    # X = X[:,:-1]
    X = zscore(X, axis = 0)
    
    # Perform SVD on the activity data to get principal components
    A = (Y - np.mean(Y, axis=0)).T
    U, S, Vt = np.linalg.svd(A, full_matrices=False)
    V = Vt.T
    
    # Eigenvalues
    var_exp_pc = [(S[i]**2) / np.sum(S**2) for i in range(len(S))]
    
    # Perform Dimensionality Reduction
    S = np.diag(S)
    # Y_pc = V[:, :PCs] @ S[:PCs, :PCs]
    Y_pc = V @ S

    # Session_ID array
    ID_array = session_id * np.ones((X.shape[0], 1))
    X_ID = np.concatenate((X, ID_array), axis = 1)
    Y_ID = np.concatenate((Y_pc, ID_array), axis = 1)

    X_reshaped = X_ID.reshape((X_ID.shape[0] // trial_len, trial_len, X_ID.shape[1])).astype(np.float32)    
    # X_reshaped = torch.from_numpy(X_reshaped)

    Y_reshaped = Y_ID.reshape((Y_ID.shape[0] // trial_len, trial_len, Y_ID.shape[1])).astype(np.float32)
    # Y_reshaped = torch.from_numpy(Y_reshaped)

    return X_reshaped, Y_reshaped

In [ ]:
# # List of Output PCs 
# out_list = [80, 55, 96, 71, 62, 47, 72, 60, 101, 65, 45, 51, 54, 51, 61, 52, 77, 61, 86, 65, 36, 52, 59, 37, 48, 59, 44, 59, 71, 89, 92, 95, 65, 100, 76, 63, 82, 43, 54, 49, 47, 103, 67]

# print(len(out_list))

In [58]:
# Try Model 

X_processed1, Y_processed1 = preprocess_data(X1,Y1, 350, 1)
X_processed2, Y_processed2 = preprocess_data(X2,Y2, 350, 2)

out_list = []
for Y in [Y_processed1, Y_processed2]:
    out_list.append(Y.shape[-1] - 1)

print(out_list)

# Create Loaders
# Test and Train split for Different Sessions
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, Dataset, random_split

# Function for creating Data Loaders
def create_dataloader(X, Y, batch_size=20):
    data = TensorDataset(torch.from_numpy(X).float(), torch.from_numpy(Y).float())
    dataloader = DataLoader(data, batch_size=batch_size, shuffle=True)
    return dataloader

# Create Loaders for YH16 
loader_YH16_1_R = create_dataloader(X_processed1, Y_processed1)
loader_YH16_2_R = create_dataloader(X_processed2, Y_processed2)

Loaders = [loader_YH16_1_R, loader_YH16_2_R]


[55, 71]


In [59]:
print(X_processed2.shape)

print(loader_YH16_2_R.dataset[:][1].shape)
print(loader_YH16_2_R.dataset[:][1].shape)


(305, 350, 56)
torch.Size([305, 350, 72])
torch.Size([305, 350, 72])


In [60]:
import random

train_loaders = Loaders[0]
test_loaders = Loaders[1]

# Extract Batches Function
def extract_batches(loader):
    batches = []
    for batch in loader: 
        batches.append(batch)
    return batches

all_batches = []
for loader in train_loaders: 
    all_batches.append(extract_batches(loader))
train_loader = DataLoader(all_batches, batch_size=1, shuffle=False)

# combined_batches = [batch for session_batches in all_batches for batch in session_batches]
# random.shuffle(combined_batches)
# train_loader = DataLoader(combined_batches, batch_size=1, shuffle=False)



In [61]:
# Example parameters
n_kp = 10
model = KeypointsNetwork(n_in=55, n_kp=n_kp, out_list=out_list)

# Forward pass
y_pred, latents = model(X_processed1)

print(f'Y_pred Shape: ', y_pred.shape)
print(f'Latents Shape: ', latents.shape)
print(f'Target Shape: ', Y_processed1.shape)

# Reshape Output and Target
y_pred = y_pred.detach().numpy().reshape((y_pred.shape[0] * y_pred.shape[1], y_pred.shape[-1]))


from sklearn.metrics import r2_score

R2 = r2_score(y_pred, Y_processed1[:,:-1])

print()
print('R2: ', R2)

TypeError: linear(): argument 'input' (position 1) must be Tensor, not numpy.ndarray

In [66]:
# Model Training 
from sklearn.metrics import r2_score
import torch.optim as optim
import time


# Connect to Device: 
# Device Object - CPU and GPU 
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Device", device)

# Instantiate Model 
model = KeypointsNetwork(n_in=55, n_kp=None, out_list=out_list)



def train_model(train_loader, model, device, num_epochs):
    import time 

    start_time = time.time()

    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.001)
    Loss_fcn = nn.MSELoss()
    best_r2 = -float('inf')
    best_loss = float('inf')

    model_save_path = f'trained_params.pth'

    train_Rsquared = []
    print(f'Training Started')

    for epoch in range(num_epochs):
        model.train()
        train_r2 = 0.0

        for X_batch, Y_batch in train_loader:
            features = X_batch.to(device).squeeze(0)
            target = Y_batch.to(device).squeeze(0)
            optimizer.zero_grad()

            target = target[:, :, :-1]
            outputs = model(features)

            loss = Loss_fcn(outputs[0], target)
            loss.backward()
            optimizer.step()
            
            outputs_np = outputs[0].detach().cpu().numpy()
            target_np = target.cpu().numpy()
            r2_batch = r2_score(target_np.reshape(-1, target_np.shape[-1]), outputs_np.reshape(-1, outputs_np.shape[-1]), multioutput='uniform_average')
            train_r2 += r2_batch

        # pc_r2 /= len(train_loader.dataset)
        train_r2 /= len(train_loader.dataset)
        train_Rsquared.append(train_r2)
        
        if loss.item() < best_loss:
            best_loss = loss.item()

        if train_r2 > best_r2:
            best_r2 = train_r2
            torch.save(model.state_dict(), model_save_path)

        if (epoch + 1) % 2 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], R2: {train_r2:.4f}')
    
    print('BEST MSE LOSS: ', best_loss)
    print('BEST R2: ', best_r2)


    end_time = time.time()
    print(f'Training Done {(end_time - start_time) / 60 :.2f} Minutes')


Device cpu


In [67]:
num_epochs = 4
train_model(train_loader, model, device, num_epochs)


model.eval()

ID = int(test_loaders.dataset[9][0][0, -1])
print(f'Test_Loader_ID:', ID)

all_targets = test_loaders.dataset[:][1][:, :, :-1]
all_features = (test_loaders.dataset[:][0][:, :, :]).to(device)

with torch.no_grad():
    all_outputs = model(all_features)
    # print(all_outputs.shape)


X_regression = all_outputs.cpu().numpy().reshape(-1, all_outputs.shape[-1])
Y_regression = all_targets.cpu().numpy().reshape(-1, all_targets.shape[-1])

BETA_OLS = np.linalg.lstsq(X_regression, Y_regression, rcond=None)
# print(BETA_OLS[0].shape)

Y_hat = X_regression @ BETA_OLS[0]
r2_pc_reg = [r2_score(Y_regression[:, i], Y_hat[:, i]) for i in range(Y_regression.shape[1])]




Training Started
Epoch [2/4], R2: 0.0012
Epoch [4/4], R2: 0.0206
BEST MSE LOSS:  396.8339538574219
BEST R2:  0.020618400736046687
Training Done 0.35 Minutes
Test_Loader_ID: 2
Unexpected Session ID: 2. Returning latents only.


In [68]:
print(len(r2_pc_reg))
print(all_outputs.shape)

71
torch.Size([305, 350, 256])
